# ¿Se necesitan más hubs M2?

Análisis para determinar si la brecha de cobertura M1-M2 se debe a:
- **A) Posicionamiento** — hubs existentes demasiado lejos de las cargas → se corrige con optimización
- **B) Hubs faltantes** — zonas sin ningún hub cercano → requieren hubs nuevos

> **Hipótesis: la brecha es de posicionamiento (A), no de hubs faltantes (B).**

### Metadata del análisis
| Campo | Valor |
|---|---|
| N registros | 57,573,489 |
| Ventana de fechas | Feb–Ago 2025 |
| Tipo de validación | Within-sample |

In [ ]:
import polars as pl
import pandas as pd
import numpy as np

OUT_M1  = 'loads_assign_m1.parquet'
OUT_M2  = 'loads_assign_m2.parquet'
HUBS_M2 = 'final_results.csv'


## 1. Cargar datos

In [7]:
m1       = pl.read_parquet(OUT_M1)
m2       = pl.read_parquet(OUT_M2)
hubs_raw = pd.read_csv(HUBS_M2)

hubs_coords = (
    hubs_raw.dropna(subset=['lat', 'lng'])
    [['lat', 'lng']].drop_duplicates().reset_index(drop=True)
)
print(f'Cargas: {len(m2):,}')
print(f'Hubs M2: {len(hubs_coords)}')

Cargas: 57,573,489
Hubs M2: 192


## 2. Cobertura global a 50 mi — M1 vs M2

In [3]:
def pct_covered(df, col_50m, col_lat):
    total     = len(df)
    con_coord = df.filter(pl.col(col_lat).is_not_null()).height
    cubiertos = df.filter(pl.col(col_50m).list.len() > 0).height
    return {
        'total':      total,
        'con_coord':  con_coord,
        'cubiertos':  cubiertos,
        '% total':    round(100 * cubiertos / total,     1),
        '% c/coord':  round(100 * cubiertos / con_coord, 1) if con_coord else 0,
    }

stats = {
    'M1 origen':  pct_covered(m1, 'hubs_50m_m1_orig', 'LatOrigin'),
    'M1 destino': pct_covered(m1, 'hubs_50m_m1_dest', 'LatDestination'),
    'M2 origen':  pct_covered(m2, 'hubs_50m_m2_orig', 'LatOrigin'),
    'M2 destino': pct_covered(m2, 'hubs_50m_m2_dest', 'LatDestination'),
}
pd.DataFrame(stats).T


,total,con_coord,cubiertos,% total,% c/coord
M1 origen,57573489.0,57046115.0,48216538.0,83.7,84.5
M1 destino,57573489.0,57038026.0,48031623.0,83.4,84.2
M2 origen,57573489.0,57046115.0,40220631.0,69.9,70.5
M2 destino,57573489.0,57038026.0,42868322.0,74.5,75.2


## 3. Cobertura M2 a múltiples radios

**Esta es la prueba clave.** Si la cobertura sube mucho al pasar de 50 a 100 mi,
significa que los hubs *existen* cerca de esas cargas — solo están mal posicionados.
Si la cobertura no sube, habría zonas sin ningún hub.

In [4]:
total = len(m2)
con_orig = m2.filter(pl.col('LatOrigin').is_not_null()).height

radios = []
for r in [50, 100, 150]:
    col = f'hubs_{r}m_m2_orig'
    cub = m2.filter(pl.col(col).list.len() > 0).height
    radios.append({
        'radio_mi': r,
        'cubiertos': cub,
        '% total':   round(100 * cub / total, 1),
        '% c/coord': round(100 * cub / con_orig, 1),
    })

df_radios = pd.DataFrame(radios).set_index('radio_mi')
print('Cobertura M2 origen por radio:')
print(df_radios.to_string())

# ¿Cuántas cargas sin cobertura a 50mi SÍ tienen hub a 100mi?
no_50  = m2.filter(pl.col('hubs_50m_m2_orig').list.len().eq(0))
si_100 = no_50.filter(pl.col('hubs_100m_m2_orig').list.len() > 0).height
print(f'\nCargas sin hub a 50mi pero CON hub a 100mi: {si_100:,}')
print(f'  → {round(100*si_100/len(no_50),1)}% de las no cubiertas tienen un hub dentro de 100mi')


Cobertura M2 origen por radio:
          cubiertos  % total  % c/coord
radio_mi                               
50         40220631     69.9       70.5
100        51440570     89.3       90.2
150        52265139     90.8       91.6

Cargas sin hub a 50mi pero CON hub a 100mi: 11,219,939
  → 92.1% de las no cubiertas tienen un hub dentro de 100mi


## 4. Distancia al hub más cercano — cargas no cubiertas a 50 mi

Para cada coordenada única sin cobertura calcula la distancia mínima a cualquier hub M2.
Si la mayoría está a < 80 mi → optimización suficiente. Si hay cola importante a > 150 mi → faltan hubs.

In [8]:
# Coordenadas únicas sin cobertura, ponderadas por número de cargas
uncov = (
    m2
    .filter(
        pl.col('LatOrigin').is_not_null() &
        pl.col('hubs_50m_m2_orig').list.len().eq(0)
    )
    .group_by(['LatOrigin', 'LngOrigin'])
    .agg(pl.len().alias('n_cargas'))
    .to_pandas()
    .reset_index(drop=True)
)
print(f'Coordenadas únicas sin cobertura: {len(uncov):,}')
print(f'Cargas representadas: {uncov["n_cargas"].sum():,}')

# Haversine vectorizado: distancia mínima a cualquier hub M2
hub_lats_r = np.radians(hubs_coords['lat'].values)
hub_lngs_r = np.radians(hubs_coords['lng'].values)
coord_lats_r = np.radians(uncov['LatOrigin'].values)
coord_lngs_r = np.radians(uncov['LngOrigin'].values)

BATCH = 5_000
min_dists = np.empty(len(uncov))
for start in range(0, len(uncov), BATCH):
    end   = min(start + BATCH, len(uncov))
    c_lat = coord_lats_r[start:end, None]
    c_lng = coord_lngs_r[start:end, None]
    dlat  = hub_lats_r - c_lat
    dlng  = hub_lngs_r - c_lng
    a     = np.sin(dlat/2)**2 + np.cos(c_lat)*np.cos(hub_lats_r)*np.sin(dlng/2)**2
    dist  = 3958.8 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    min_dists[start:end] = dist.min(axis=1)

uncov['dist_mi'] = min_dists
media_w = np.average(min_dists, weights=uncov['n_cargas'])
print(f'\nDistancia media ponderada al hub más cercano: {media_w:.1f} mi')
print(f'Mediana: {np.median(min_dists):.1f} mi')


Coordenadas únicas sin cobertura: 70,852
Cargas representadas: 12,175,867

Distancia media ponderada al hub más cercano: 55.8 mi
Mediana: 55.2 mi


In [9]:
# Distribución ponderada por cargas
bins   = [0, 60, 70, 80, 100, 120, 150, 200, 300, 9999]
labels = ['<60', '60-70', '70-80', '80-100', '100-120', '120-150', '150-200', '200-300', '>300']

uncov['dist_bin'] = pd.cut(uncov['dist_mi'], bins=bins, labels=labels, right=False)

cargas_bin = uncov.groupby('dist_bin', observed=True)['n_cargas'].sum()
coords_bin = uncov.groupby('dist_bin', observed=True).size()
pct        = (cargas_bin / cargas_bin.sum() * 100).round(1)
acum       = pct.cumsum().round(1)

tabla = pd.DataFrame({'coords': coords_bin, 'cargas': cargas_bin, '%': pct, '% acum': acum})
print(tabla.to_string())

pct_lt80  = uncov.loc[uncov['dist_mi'] < 80,  'n_cargas'].sum() / uncov['n_cargas'].sum() * 100
pct_lt100 = uncov.loc[uncov['dist_mi'] < 100, 'n_cargas'].sum() / uncov['n_cargas'].sum() * 100
pct_gt150 = uncov.loc[uncov['dist_mi'] >= 150, 'n_cargas'].sum() / uncov['n_cargas'].sum() * 100
print(f'\n% cargas < 80 mi:  {pct_lt80:.1f}%')
print(f'% cargas < 100 mi: {pct_lt100:.1f}%')
print(f'% cargas >= 150 mi: {pct_gt150:.1f}%')


          coords   cargas     %  % acum
dist_bin                               
<60        40956  6978215  57.3    57.3
60-70      10886  2052089  16.9    74.2
70-80       7818  1437118  11.8    86.0
80-100      7480  1219244  10.0    96.0
100-120     2473   282979   2.3    98.3
120-150      973   169049   1.4    99.7
150-200      263    37169   0.3   100.0
200-300        2        3   0.0   100.0
>300           1        1   0.0   100.0

% cargas < 80 mi:  86.0%
% cargas < 100 mi: 96.0%
% cargas >= 150 mi: 0.3%


## 5. Veredicto

In [10]:
total      = len(m2)
no_50_cnt  = m2.filter(pl.col('hubs_50m_m2_orig').list.len().eq(0)).height
no_100_cnt = m2.filter(
    pl.col('LatOrigin').is_not_null() &
    pl.col('hubs_100m_m2_orig').list.len().eq(0)
).height
no_150_cnt = m2.filter(
    pl.col('LatOrigin').is_not_null() &
    pl.col('hubs_150m_m2_orig').list.len().eq(0)
).height

print('═' * 55)
print('VEREDICTO: ¿Se necesitan más hubs M2?')
print('═' * 55)
print(f'Cargas sin hub a  50 mi: {no_50_cnt:>10,}  ({100*no_50_cnt/total:.1f}%)')
print(f'Cargas sin hub a 100 mi: {no_100_cnt:>10,}  ({100*no_100_cnt/total:.1f}%)')
print(f'Cargas sin hub a 150 mi: {no_150_cnt:>10,}  ({100*no_150_cnt/total:.1f}%)')
print()

# Criterio: si > 90% de las no cubiertas a 50mi tienen hub a 100mi → es posicionamiento
no_50_df   = m2.filter(pl.col('hubs_50m_m2_orig').list.len().eq(0))
si_100_cnt = no_50_df.filter(pl.col('hubs_100m_m2_orig').list.len() > 0).height
pct_si_100 = 100 * si_100_cnt / len(no_50_df) if len(no_50_df) else 0

print(f'De las cargas no cubiertas a 50mi:')
print(f'  - {pct_si_100:.1f}% tienen un hub M2 dentro de 100mi')
print(f'  - {100-pct_si_100:.1f}% no tienen ningún hub dentro de 100mi')
print()

if pct_si_100 >= 85:
    print('→ DIAGNÓSTICO: La brecha es de POSICIONAMIENTO.')
    print('  La mayoría de cargas no cubiertas tienen un hub cerca (< 100mi).')
    print('  Optimizar centroides con SEARCH_RADIUS=80 debería cerrar gran parte de la brecha.')
    print('  NO se necesitan hubs nuevos.')
else:
    print('→ DIAGNÓSTICO: Hay zonas sin cobertura real.')
    print(f'  {100-pct_si_100:.1f}% de cargas no cubiertas están a >100mi de cualquier hub.')
    print('  Evaluar si agregar hubs en esas zonas.')


═══════════════════════════════════════════════════════
VEREDICTO: ¿Se necesitan más hubs M2?
═══════════════════════════════════════════════════════
Cargas sin hub a  50 mi: 12,175,867  (21.1%)
Cargas sin hub a 100 mi:    955,928  (1.7%)
Cargas sin hub a 150 mi:    131,359  (0.2%)

De las cargas no cubiertas a 50mi:
  - 92.1% tienen un hub M2 dentro de 100mi
  - 7.9% no tienen ningún hub dentro de 100mi

→ DIAGNÓSTICO: La brecha es de POSICIONAMIENTO.
  La mayoría de cargas no cubiertas tienen un hub cerca (< 100mi).
  Optimizar centroides con SEARCH_RADIUS=80 debería cerrar gran parte de la brecha.
  NO se necesitan hubs nuevos.
